# 05 · Agent / Controller: the Intelligent Retail Decision Assistant

A **hybrid, inspectable controller** (`src/agent.py`). Its routing is transparent: every decision is written to a trace.

| Tool | What it does | Grounding |
|---|---|---|
| `product_search` | RAG recommendation (notebook 03) | ids must be retrieved; budget enforced |
| `policy_qa` | policy answers (notebook 04) | doc_id citation + verbatim quote, or refusal |
| `segment_insights` | analytics from the RFM segments (notebook 02) | numbers read from `segment_summary.json`, no LLM |
| `review_summary` | what customers say about a product type | counts computed; summary only from review texts |
| `summarise_products` | explain the top 3 of the previous search | session memory, only those 3 ids |
| `calculator` | discounts and arithmetic | safe AST evaluation |

**Routing steps:**
1. **Guardrails** stop empty or out-of-scope input.
2. **Rule-based intent scores** come from regex cues, with specificity rules: for example, "watch" inside a warranty question is not a shopping request.
3. A **multi-intent** query runs up to `MAX_TOOL_CALLS = 3` tools, and each tool receives only its own clause.
4. If no rule fires, an **LLM router** chooses one tool from the fixed list, as JSON, and its choice is validated against the list.

In [ ]:
# --- setup: make the project package importable and show the run mode ---
import sys, os, json
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
import src                                   # sets macOS OpenMP compatibility flags
from src import config
from src.llm import mode_label
import pandas as pd, numpy as np
import matplotlib.pyplot as plt, seaborn as sns
pd.set_option("display.max_colwidth", 110); pd.set_option("display.width", 220)
sns.set_theme(style="whitegrid", palette="Set2"); plt.rcParams["figure.dpi"] = 110
OUT = config.OUTPUTS
print("Mode:", mode_label(), "| outputs ->", OUT.relative_to(ROOT))

In [ ]:
from src.agent import RetailAssistant, route, score_intents, CUES, TOOLS, SUPPRESS
pd.DataFrame({"tool": list(CUES), "cue patterns": [", ".join(v) for v in CUES.values()]})

## 1. Routing decisions only (no tools executed)

In [ ]:
probe = ["Recommend casual sneakers under ₹4000 for daily wear.", "Can I return a discounted product after 7 days?",
         "Is physical damage to my watch covered under the warranty?", "Which customer segment seems most price-sensitive?",
         "Show me a blue backpack for travel under 3000 and tell me if I can exchange it later.",
         "What do customers say about Nike running shoes?", "Summarise the top three retrieved products.",
         "What is 20% off 2499?", "Tell me about the weather today", "Do you offer gift wrapping?"]
rt = []
for q in probe:
    tools, trace = route(q)
    rt.append({"query": q, "selected": tools, "why": " | ".join(trace)})
routes = pd.DataFrame(rt); routes.to_csv(OUT / "05_routing_table.csv", index=False); routes

*"Do you offer gift wrapping?"* matches no rule:
- **Live mode:** the LLM router decides, typically choosing `policy_qa`, which then **refuses** because no policy covers gift wrapping.
- **Offline mode:** the assistant asks the user to rephrase.

## 2. Full assistant demo with traces

This covers the 5 example queries from the brief plus extra cases.

In [ ]:
bot = RetailAssistant()
demo = ["Recommend casual sneakers under ₹4000 for daily wear.",
        "Can I return a discounted product after 7 days?",
        "Suggest products for a premium customer who prefers black formal shoes.",
        "Which customer segment seems most price-sensitive?",
        "Summarise the top three retrieved products and explain why they fit the query.",
        "Show me a blue backpack for travel under 3000 and tell me if I can exchange it later.",
        "What do customers say about Nike running shoes?",
        "Is physical damage to my watch covered under the warranty?",
        "What is 20% off 2499?",
        "What's the weather in Mumbai today?"]
outs = [bot.ask(q) for q in demo]

In [ ]:
with open(OUT / "05_demo_transcript.jsonl", "w") as f:
    for o in outs: f.write(json.dumps(o, default=str, ensure_ascii=False) + "\n")
pd.DataFrame([{"query": o["query"], "tools": o["tools"], "latency_s": o["latency_s"],
               "first line of answer": o["final_answer"].splitlines()[0][:90]} for o in outs])

**What the traces show:**
- **Single-intent queries** each hit exactly one tool.
- **Multi-intent:** the backpack + exchange query runs `product_search` on *"blue backpack … under 3000"* and `policy_qa` on *"can I exchange it later"*, then combines the two answers.
- **Session memory:** "summarise the top three" reuses the previous search. It does not search again.
- **Specificity:** "watch … warranty" goes only to `policy_qa`.
- **Guardrail:** "weather" never reaches a tool.
- **Bounded:** at most 3 tool calls per query, each with its own status and latency in the trace.